A notebook to test the functions of the new vrdr_resampler Python module.

In [ ]:
# auto reload
%load_ext autoreload
%autoreload 2

In [ ]:
import vrdr_resampler as vrdr
from pathlib import Path

# Loading the CaSSIS Spectral Response

Setting a string id for the instrument, that will prefixed to output files:

In [ ]:
inst_id = 'c'

Setting the path to the instrument file:

In [ ]:
# instrument path
inst_path = Path('instrument_response', 'CaSSIS-2019.csv')

Loading instrument transmission profiles:

In [ ]:
cassis_data = vrdr.load_channel_profiles(inst_path)
band_arr, arr_wvls, band_list, band_cwls = vrdr.unpack_channel_profiles(cassis_data)

Checking the profile spectral range and resolution:

In [ ]:
fig, ax = vrdr.plot_channel_profiles(cassis_data, instrument='CaSSIS-2019')

# Loading a VRDR Tile

Setting the data directory:

In [ ]:
vrdr_path = Path('/Volumes', 'RS_STORE', 'vrdr')

Setting the output directory:

In [ ]:
output_path = vrdr_path.parent / 'vrdr_cassis'
output_path.mkdir(exist_ok=True) 

Finding the quadrant directories:

In [ ]:
mc_paths = vrdr.get_quadrant_dirs(vrdr_path)
mc_path = mc_paths[0]  # Example: process the first quadrant directory

Finding the tile paths in the quadrant directory:

In [ ]:
lbl_paths = vrdr.get_tile_lbls(mc_path)
lbl_path = lbl_paths[0]  # Example: process the first label file

Preparing the output file directory:

In [ ]:
out_dir = vrdr.prepare_output_dir(lbl_path, mc_path, output_path, overwrite=True)
out_dir

Loading the tile, getting the band centre-wavelengths, and the no-data value:

In [ ]:
vrdr_cube, vrdr_cwls, no_data = vrdr.load_vrdr_tile(lbl_path)

# Resampling to the Spectral Response to a CaSSIS I/F Cube

Resampling the VRDR Cube from the VRDR CWLs to the instrument bands:

In [ ]:
cifcub = vrdr.resample_cube(band_arr, arr_wvls, vrdr_cube, vrdr_cwls, no_data)

# Exporting the I/F Cube

In [ ]:
out_file = vrdr.export_ifcub_geotiff(cifcub,band_list,band_cwls,no_data,inst_id,lbl_path,out_dir)
out_hdr = vrdr.write_ifcub_envi_hdr(band_list, band_cwls, no_data,[4,2,1], inst_id, lbl_path, out_dir)

## Verifying the Resampling Method

Checking the interpolation of the band wavelengths to the VRDR CWLs:

In [ ]:
b_w = arr_wvls
c_w = vrdr_cwls
b_a = band_arr
resampled_profiles = [vrdr.interp_band_profile(b_p, b_w, c_w) for b_p in b_a.T]

In [ ]:
import numpy as np
import pandas as pd
resampled_inst = pd.DataFrame(data=np.array(resampled_profiles).T, columns=band_list, index=c_w)

In [ ]:
fig, ax = vrdr.plot_channel_profiles(resampled_inst, instrument='CaSSIS-2019 Resampled')

Checking a random selection of VRDR tile spectra

In [ ]:
# select random locations in the last 2 dimensions of the cube
import random
import matplotlib.pyplot as plt

num_samples = 10
x_size, y_size = vrdr_cube.shape[1], vrdr_cube.shape[2]
random_coords = [(random.randint(0, x_size - 1), random.randint(0, y_size - 1)) for _ in range(num_samples)]

fig, ax = plt.subplots(figsize=(5, 4))

y_max = 0.4

for band in resampled_inst.columns:
    ax.plot(resampled_inst.index, y_max*resampled_inst[band]/np.nanmax(resampled_inst[band]), label=band, linestyle='-',marker='o', alpha=0.2, markersize=1)
    # fill the area under each band profile
    ax.fill_between(resampled_inst.index, 0, y_max*resampled_inst[band]/np.nanmax(resampled_inst[band]), alpha=0.1)

for x, y in random_coords:
    vrdr_spectrum = vrdr_cube[:, x, y]
    cassis_spectrum = cifcub[:, x, y]

    ax.plot(vrdr_cwls, vrdr_spectrum, label=f'{x,y}', marker='o', markersize=3, linestyle='None', alpha=0.5)
    # get colour used in the plot above
    color = ax.get_lines()[-1].get_color()
    ax.plot(band_cwls, cassis_spectrum, label=f'{x,y}', marker='x', markersize=6, color=color)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Reflectance')
# plt.legend()
ax.set_title(f'Randomly selected pixel spectra')
ax.set_ylim(0, y_max)
ax.grid()
fig.show()

Note that data points with wavelengths <400 nm (i.e. the apparently noisy channels) are 0-weighted by the CaSSIS BLU transmission, so they effectively are not counted.

# Prepare Products

Import the instrument specific product class:

In [ ]:
import cassis_products as cassis

## Spectral Parameter (or Summary Product) Stack

Make a stack of the spectral parameters:

In [ ]:
cspcub, sp_ids = cassis.spectral_parameter_stack(cifcub, no_data)

Export the stack to a GeoTIFF:

In [ ]:
vrdr.export_spcub_geotiff(cspcub,sp_ids,no_data,inst_id,lbl_path,out_dir)

In [ ]:
vrdr.write_spcub_envi_hdr(sp_ids, no_data, [3, 1, 2],inst_id,lbl_path,out_dir)

## Browse Products

Stretching the I/F Cube: stretching float I/F data to 8-bit unsigned int for efficient browse product storage involves stretching each channel.

The standard method for CaSSIS is to use percentile stretches, where the bottom 0.1% and the top 99.95% percentiles are used to set the clip values.

We log these stretch parameters.

In [ ]:
st_cub, st_params = vrdr.stretch_cube(cifcub, no_data)

In [ ]:
log_path = vrdr.log_st_params(st_params, band_list, lbl_path, out_dir)

In [ ]:
cifcub_mod = cassis.cassis_add_sblu(cifcub, no_data)
st_cub, st_params = vrdr.stretch_cube(cifcub_mod, no_data)
for bp_id in cassis.BP_LIST:
    bp_func = cassis.browse_product(bp_id)
    bp, bp_bands = bp_func(st_cub)
    vrdr.export_bp_geotiff(bp, bp_id, 255, bp_bands, inst_id, lbl_path, out_dir)
    plt.imshow(bp)
    plt.title(f'CaSSIS {bp_id} Browse Product')
    plt.show()

## Composite Spectral Parameter Browse Products

In [ ]:
for cp_id in cassis.CP_LIST:
    cp_func = cassis.composite_product(cp_id)
    cp, cp_list = cp_func(cifcub, no_data)
    cp_st, st_params = vrdr.stretch_cube(cp, no_data)
    vrdr.export_bp_geotiff(cp_st.transpose((1, 2, 0)), cp_id, 255, cp_list, inst_id, lbl_path, out_dir)
    plt.imshow(cp_st.transpose((1, 2, 0)))
    plt.title(f'CaSSIS {cp_id} Composite Browse Product')
    plt.show()

# Running the Loop

In [ ]:
from tqdm.auto import tqdm

In [ ]:
vrdr_path = Path('/Volumes', 'RS_STORE', 'vrdr')
output_path = vrdr_path.parent / 'vrdr_cassis'
output_path.mkdir(exist_ok=True) 

inst_id = 'c'
inst_path = Path('instrument_response', 'CaSSIS-2019.csv')
cassis_data = vrdr.load_channel_profiles(inst_path)
band_arr, arr_wvls, band_list, band_cwls = vrdr.unpack_channel_profiles(cassis_data)

mc_paths = vrdr.get_quadrant_dirs(vrdr_path)

for mc_path in tqdm(mc_paths, desc='Quadrants'):
    # if mc_path.stem not in ['mc16']:
    #     continue
    print('Processing MC Directory: ', mc_path.name)
    # get lbl files but ignore hidden files
    lbl_paths = vrdr.get_tile_lbls(mc_path)
    # sort

    for lbl_path in tqdm(lbl_paths, desc=f'{mc_path.stem} Tiles'):
        print(' - ', lbl_path.stem)

        out_dir = vrdr.prepare_output_dir(lbl_path, mc_path, output_path, overwrite=False)
        if out_dir == Path():
            continue

        vrdr_cube, vrdr_wvls, no_data = vrdr.load_vrdr_tile(lbl_path)

        cifcub = vrdr.resample_cube(band_arr, arr_wvls, vrdr_cube, vrdr_wvls, no_data)

        out_file = vrdr.export_ifcub_geotiff(cifcub, band_list, band_cwls, no_data, inst_id, lbl_path, out_dir)
        out_hdr = vrdr.write_ifcub_envi_hdr(band_list, band_cwls, no_data, [4, 2, 1], inst_id, lbl_path, out_dir)

        st_cub, st_params = vrdr.stretch_cube(cifcub, no_data)
        log_path = vrdr.log_st_params(st_params, band_list, lbl_path, out_dir)

        cspcub, sp_ids = cassis.spectral_parameter_stack(cifcub, no_data)
        vrdr.export_spcub_geotiff(cspcub,sp_ids,no_data,inst_id,lbl_path,out_dir)
        vrdr.write_spcub_envi_hdr(sp_ids, no_data, [3, 1, 2],inst_id,lbl_path,out_dir)

        cifcub_mod = cassis.cassis_add_sblu(cifcub, no_data)
        st_cub, st_params = vrdr.stretch_cube(cifcub_mod, no_data)
        for bp_id in cassis.BP_LIST:
            bp_func = cassis.browse_product(bp_id)
            bp, bp_bands = bp_func(st_cub)
            vrdr.export_bp_geotiff(bp, bp_id, 255, bp_bands, inst_id, lbl_path, out_dir)

        for cp_id in cassis.CP_LIST:
            cp_func = cassis.composite_product(cp_id)
            cp, cp_list = cp_func(cifcub, no_data)
            cp_st, st_params = vrdr.stretch_cube(cp, no_data)
            vrdr.export_bp_geotiff(cp_st.transpose((1, 2, 0)), cp_id, 255, cp_list, inst_id, lbl_path, out_dir)
        

Now I'd like a quick script that loops over all processed tiles in a quadrant, gets the quadrant-wide stretch parameters, and then makes a mosaic browse product at that level.

# Making Quadrant Wide Mosaics

Let's re-write this from scratch, now loading in the .log file from each porcessed mosaic, and putting the new stretch products in a dedicated new product directory (vrdr_cassis_quads).

In [ ]:
# get cassis quadrant directory paths
vrdr_cassis_path = Path('/Volumes', 'RS_STORE', 'vrdr_cassis')
vrdr_path = Path('/Volumes', 'RS_STORE', 'vrdr') # still need to go back and get the orig. label files
output_path = vrdr_cassis_path.parent / 'vrdr_cassis_quads'
output_path.mkdir(exist_ok=True) 

mc_paths = vrdr.get_quadrant_dirs(vrdr_cassis_path)

for mc_path in tqdm(mc_paths, desc='Quadrants'):
    print('Processing MC Directory: ', mc_path.name)
    tile_paths = vrdr.get_tile_dirs(mc_path)

    quad_st_params = vrdr.get_quadrant_stretch_params(tile_paths)

    for tile_path in tqdm(tile_paths, desc='Tiles'):
        print(' - ', tile_path.stem)
        # get tiff files
        orig_tile_path = vrdr_path / mc_path.stem / tile_path.stem
        lbl_path = Path(orig_tile_path).with_suffix('.lbl')

        out_dir = output_path / mc_path.stem
        out_dir.mkdir(parents=True, exist_ok=True)

        # read the cifcub
        cube_file = [f for f in tile_path.glob('*cifcub*.tif')][0]
        cifcub, no_data = vrdr.read_geotiff_cube(cube_file)

        st_cub, st_params = vrdr.stretch_cube(cifcub, no_data, st_params=quad_st_params)
        
        bp_list = cassis.BP_LIST.copy()
        bp_list.remove('RGB')
        for bp_id in bp_list:
            bp, bp_bands = cassis.browse_product(bp_id)(st_cub)
            vrdr.export_bp_geotiff(bp, bp_id, 255, bp_bands, inst_id, lbl_path, out_dir/bp_id, 'quadrant')

    log_file = vrdr.log_st_params(quad_st_params, band_list, lbl_path, out_dir)
